# FC Box Plots: Models x Conditions x Regions

2x2 grid of box plots (one per brain region) comparing FC across 3 pharmacological conditions and 6 models (5 trained + empirical).

In [1]:
import os
import sys
from pathlib import Path

notebook_dir = Path().absolute()
if notebook_dir.name == 'examples':
    os.chdir(notebook_dir.parent)

import torch
import numpy as np
import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline

## Configuration

In [2]:
LSD_DATA_DIR   = "data/lsd"
CHECKPOINT_DIR = "checkpoints"
DATASET_TYPE   = "lsd"
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
N_STEPS        = 240          # simulation length (match empirical timepoints)
GROUP_SIZE     = 0            # 0 = per-subject FC

ROI_NAMES = ["Thalamus", "Visual Cortex", "PCC", "Temporal Gyrus"]

CONDITION_LABELS = {0: "Placebo", 1: "LSD+Ketanserin", 2: "LSD"}
CONDITION_COLORS = {0: "#4878CF", 1: "#D65F5F", 2: "#6ACC65"}

print(f"Device: {DEVICE}")

Device: cpu


## 1. Load Dataset

In [3]:
from src.dataset import NeuroscienceDataset
from src.metrics import compute_static_fc, fisher_batch_average
from src.training.evaluation import rollout_model

dataset = NeuroscienceDataset.from_lsd(data_dir=LSD_DATA_DIR, normalize=True, device=DEVICE)
print(f"n_rois={dataset.n_rois}  n_subjects={dataset.n_subjects}  T={dataset.n_timepoints}")

# Group subject indices by condition
subjects_per_ctrl = {}
for cv in sorted(CONDITION_LABELS.keys()):
    mask = dataset.control[:, 0] == cv
    if mask.sum() > 0:
        subjects_per_ctrl[cv] = mask.nonzero(as_tuple=True)[0]
        print(f"  ctrl={cv} ({CONDITION_LABELS[cv]}): {mask.sum().item()} subjects")

LSD dataset: conditions=['Placebo', 'LSD', 'LSD+Ketanserin'], control_values=[0, 1, 2], n_patients=25, n_timeseries=75
n_rois=4  n_subjects=75  T=240
  ctrl=0 (Placebo): 25 subjects
  ctrl=1 (LSD+Ketanserin): 25 subjects
  ctrl=2 (LSD): 25 subjects


## 2. Load Checkpoints

In [4]:
from src.models import load_model_from_checkpoint

_MODEL_TAGS = [
    ("gnn_hopf",      "GNN Hopf"),
    ("hybrid_neural", "Hybrid+Neural"),
    ("hybrid_hopf",   "Hybrid Hopf"),
    ("nsde",          "Neural SDE"),
    ("hopf",          "Hopf"),
]

def _short_label(stem):
    lower = stem.lower()
    for tag, label in _MODEL_TAGS:
        if tag in lower:
            return label + (" (Grid)" if "grid" in lower else "")
    return stem

ckpt_dir = Path(CHECKPOINT_DIR)
checkpoint_paths = sorted(ckpt_dir.glob(f"*{DATASET_TYPE}*.pt"))
print(f"Found {len(checkpoint_paths)} checkpoints")

models = {}
for p in checkpoint_paths:
    try:
        model, mtype, _ = load_model_from_checkpoint(str(p), device=DEVICE)
        if model.n_rois != dataset.n_rois:
            print(f"  Skip {p.name}: ROI mismatch")
            continue
        label = _short_label(p.stem)
        models[label] = model
        print(f"  Loaded '{label}' ({mtype})")
    except Exception as exc:
        print(f"  Failed {p.name}: {exc}")

print(f"\n{len(models)} model(s): {list(models.keys())}")

Found 6 checkpoints
  Loaded 'GNN Hopf' (GNNHopfModel)
  Loaded 'Hopf' (CoupledHopfModel)
  Loaded 'Hopf (Grid)' (CoupledHopfModel)
  Loaded 'Hybrid Hopf' (HybridHopfModel)
  Loaded 'Hybrid+Neural' (HybridHopfNeuralModel)
  Loaded 'Neural SDE' (NeuralSDE)

6 model(s): ['GNN Hopf', 'Hopf', 'Hopf (Grid)', 'Hybrid Hopf', 'Hybrid+Neural', 'Neural SDE']


## 3. Compute Per-Subject FC for Each Model and Condition\n\nFor each condition we select the corresponding subjects, simulate (or use empirical timeseries), and compute per-subject FC matrices. We then extract the mean connectivity of each ROI (mean of its row, excluding diagonal).

In [5]:
n_rois = dataset.n_rois
sim_steps = min(N_STEPS, dataset.n_timepoints)

def mean_fc_per_roi(fc_matrices):
    """Per-subject mean FC for each ROI (mean of row, excluding diagonal).
    
    Args:
        fc_matrices: (n_subjects, n_rois, n_rois) tensor
    Returns:
        (n_subjects, n_rois) numpy array
    """
    fc = fc_matrices.detach().cpu().numpy()
    if np.iscomplexobj(fc):
        fc = fc.real
    mask = ~np.eye(fc.shape[-1], dtype=bool)
    # For each subject, for each ROI, take mean of off-diagonal entries in that row
    result = np.zeros((fc.shape[0], fc.shape[1]))
    for i in range(fc.shape[1]):
        result[:, i] = fc[:, i, mask[i]].mean(axis=-1)
    return result

# Collect data: fc_data[model_name][condition] = (n_subjects, n_rois)
fc_data = {}

# --- Empirical ---
fc_data["Empirical"] = {}
for cv, idx in subjects_per_ctrl.items():
    emp_ts = dataset.timeseries[idx, :, :sim_steps]
    emp_fc = compute_static_fc(emp_ts)  # (n_subj, n_rois, n_rois)
    fc_data["Empirical"][cv] = mean_fc_per_roi(emp_fc)
    print(f"  Empirical ctrl={cv}: {emp_fc.shape[0]} subjects")

# --- Trained models ---
for mname, model in models.items():
    fc_data[mname] = {}
    for cv, idx in subjects_per_ctrl.items():
        target_ts = dataset.timeseries[idx, :, :sim_steps]
        initial_state = target_ts[:, :, 0]
        n_subj = len(idx)

        # Build control tensor if model uses control
        ctrl = None
        if getattr(model, "n_control_dims", 0) > 0:
            ctrl = torch.full((n_subj, 1), float(cv), dtype=torch.float32, device=DEVICE)

        with torch.no_grad():
            simulated = rollout_model(model, initial_state, sim_steps, dataset.dt, control=ctrl)
            fc_pred = compute_static_fc(simulated)  # (n_subj, n_rois, n_rois)

        fc_data[mname][cv] = mean_fc_per_roi(fc_pred)
        print(f"  {mname} ctrl={cv}: {fc_pred.shape[0]} subjects")

model_names = list(fc_data.keys())
print(f"\nModels: {model_names}")

  Empirical ctrl=0: 25 subjects
  Empirical ctrl=1: 25 subjects
  Empirical ctrl=2: 25 subjects
  GNN Hopf ctrl=0: 25 subjects
  GNN Hopf ctrl=1: 25 subjects
  GNN Hopf ctrl=2: 25 subjects
  Hopf ctrl=0: 25 subjects
  Hopf ctrl=1: 25 subjects
  Hopf ctrl=2: 25 subjects
  Hopf (Grid) ctrl=0: 25 subjects
  Hopf (Grid) ctrl=1: 25 subjects
  Hopf (Grid) ctrl=2: 25 subjects
  Hybrid Hopf ctrl=0: 25 subjects
  Hybrid Hopf ctrl=1: 25 subjects
  Hybrid Hopf ctrl=2: 25 subjects
  Hybrid+Neural ctrl=0: 25 subjects
  Hybrid+Neural ctrl=1: 25 subjects
  Hybrid+Neural ctrl=2: 25 subjects
  Neural SDE ctrl=0: 25 subjects
  Neural SDE ctrl=1: 25 subjects
  Neural SDE ctrl=2: 25 subjects

Models: ['Empirical', 'GNN Hopf', 'Hopf', 'Hopf (Grid)', 'Hybrid Hopf', 'Hybrid+Neural', 'Neural SDE']


## 4. Box Plots: 2x2 Grid (one per region)

In [6]:
ctrl_vals = sorted(CONDITION_LABELS.keys())
n_models = len(model_names)
n_conds = len(ctrl_vals)

# Box positions: group by model, sub-group by condition
box_width = 0.25
group_width = n_conds * box_width + 0.15  # spacing between model groups

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for roi_idx in range(n_rois):
    ax = axes[roi_idx]
    all_bp = []

    for mi, mname in enumerate(model_names):
        group_center = mi * group_width
        for ci, cv in enumerate(ctrl_vals):
            data = fc_data[mname].get(cv)
            if data is None:
                continue
            values = data[:, roi_idx]
            pos = group_center + (ci - 1) * box_width
            color = CONDITION_COLORS[cv]

            bp = ax.boxplot(
                values, positions=[pos], widths=box_width * 0.8,
                patch_artist=True, manage_ticks=False,
                boxprops=dict(facecolor=color, alpha=0.7),
                medianprops=dict(color="black", linewidth=1.5),
                whiskerprops=dict(color="gray"),
                capprops=dict(color="gray"),
                flierprops=dict(marker="o", markersize=3, alpha=0.5),
            )
            all_bp.append(bp)

    # X-axis: model names at group centers
    ax.set_xticks([mi * group_width for mi in range(n_models)])
    ax.set_xticklabels(model_names, rotation=25, ha="right", fontsize=9)
    ax.set_ylabel("Mean FC (off-diagonal)", fontsize=10)
    ax.set_title(ROI_NAMES[roi_idx], fontsize=13, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")

# Shared legend
legend_patches = [
    mpatches.Patch(facecolor=CONDITION_COLORS[cv], alpha=0.7, label=CONDITION_LABELS[cv])
    for cv in ctrl_vals
]
fig.legend(handles=legend_patches, loc="upper center", ncol=n_conds, fontsize=11,
           bbox_to_anchor=(0.5, 1.0), frameon=True)

fig.suptitle("FC by Region, Model, and Condition", fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("paper/images/lsd/fc_boxplots_by_region.pdf", bbox_inches="tight", dpi=150)
plt.show()

## 5. Summary Table (Mean +/- Std)

In [7]:
import pandas as pd

rows = []
for mname in model_names:
    for cv in ctrl_vals:
        data = fc_data[mname].get(cv)
        if data is None:
            continue
        for roi_idx, roi_name in enumerate(ROI_NAMES):
            vals = data[:, roi_idx]
            rows.append({
                "Model": mname,
                "Condition": CONDITION_LABELS[cv],
                "Region": roi_name,
                "Mean FC": f"{vals.mean():.3f}",
                "Std FC": f"{vals.std():.3f}",
                "N": len(vals),
            })

df = pd.DataFrame(rows)
df

,Model,Condition,Region,Mean FC,Std FC,N
0,Empirical,Placebo,Thalamus,0.287,0.123,25
1,Empirical,Placebo,Visual Cortex,0.232,0.183,25
2,Empirical,Placebo,PCC,0.313,0.184,25
3,Empirical,Placebo,Temporal Gyrus,0.236,0.161,25
4,Empirical,LSD+Ketanserin,Thalamus,0.182,0.137,25
...,...,...,...,...,...,...
79,Neural SDE,LSD+Ketanserin,Temporal Gyrus,0.039,0.062,25
80,Neural SDE,LSD,Thalamus,0.250,0.036,25
81,Neural SDE,LSD,Visual Cortex,0.220,0.051,25
82,Neural SDE,LSD,PCC,0.236,0.050,25


## 6. Alternative: Conditions on X-axis, Models as Colors

In [8]:
import matplotlib.cm as cm

# One color per model
model_colors = {
    name: cm.tab10(i / max(len(model_names) - 1, 1))
    for i, name in enumerate(model_names)
}

box_width = 0.08
group_width = len(model_names) * box_width + 0.15   # spacing between condition groups

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.flatten()

for roi_idx in range(n_rois):
    ax = axes[roi_idx]

    for ci, cv in enumerate(ctrl_vals):
        group_center = ci * group_width
        offset_start = -(len(model_names) - 1) / 2 * box_width

        for mi, mname in enumerate(model_names):
            data = fc_data[mname].get(cv)
            if data is None:
                continue
            values = data[:, roi_idx]
            pos = group_center + offset_start + mi * box_width
            color = model_colors[mname]

            ax.boxplot(
                values, positions=[pos], widths=box_width * 0.85,
                patch_artist=True, manage_ticks=False,
                boxprops=dict(facecolor=color, alpha=0.75),
                medianprops=dict(color="black", linewidth=1.5),
                whiskerprops=dict(color="dimgray"),
                capprops=dict(color="dimgray"),
                flierprops=dict(marker="o", markersize=3, alpha=0.4,
                                markerfacecolor=color, markeredgewidth=0),
            )

    # Condition names on x-axis
    ax.set_xticks([ci * group_width for ci in range(n_conds)])
    ax.set_xticklabels([CONDITION_LABELS[cv] for cv in ctrl_vals], fontsize=10)
    ax.set_ylabel("Mean FC (off-diagonal)", fontsize=10)
    ax.set_title(ROI_NAMES[roi_idx], fontsize=13, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")

# Shared legend — one entry per model
legend_patches = [
    mpatches.Patch(facecolor=model_colors[mname], alpha=0.75, label=mname)
    for mname in model_names
]
fig.legend(handles=legend_patches, loc="upper center",
           ncol=min(len(model_names), 4), fontsize=9,
           bbox_to_anchor=(0.5, 1.01), frameon=True)

fig.suptitle("FC by Region — Models grouped within each Condition",
             fontsize=14, fontweight="bold", y=1.05)
plt.tight_layout()
plt.savefig("paper/images/lsd/fc_boxplots_by_region_v2.pdf", bbox_inches="tight", dpi=150)
plt.show()